<a href="https://colab.research.google.com/github/Kinds-of-Intelligence-CFI/measurement-layouts/blob/main/basic_measurement_layout/basic_measurement_layout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Measurement Layout

In [1]:
!pip install pymc --quiet
!pip install arviz --quiet

In [2]:
import arviz as az
import graphviz
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import random as rm
import requests
import seaborn as sns
import itertools

from io import StringIO
from collections import defaultdict
from IPython.display import Image
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from pymc import model

print(f"Running on PyMC v{pm.__version__}")

Running on PyMC v5.26.1


In [3]:
# Base URL for raw GitHub files
base_url = "https://raw.githubusercontent.com/Kinds-of-Intelligence-CFI/measurement-layouts/refs/heads/main/basic_measurement_layout/data/"

# Pixels and noise levels
pixel_sizes = range(4, 44, 4)
noise_levels = range(0, 10)

# Read and combine
dataframes = []
for p in pixel_sizes:
  for n in noise_levels:
    url = base_url + f"results_pixels_{p}_noise_0.{n}.csv"
    response = requests.get(url)
    df = pd.read_csv(StringIO(response.text))
    dataframes.append(df)

# Combine all DataFrames
combined_df = pd.concat(dataframes, ignore_index=True)

In [4]:
def logistic_general(x, min, max, c = 0, p = 0.99):

  """
  Generalized version of the logistic where min can be any number, not just 0 as above.

  :param min: The min ability/demand
  :param max: The max ability/demand
  :param c: Set different from 0 when we are dealing with multiple choice responses; e.g. c is 0.5 when the response is yes/no, and c is 0.25 if there are 4 possible choices
  :param p: The probability we want to set the max margin to (e.g. 0.99 or 0.999); the min margin will be set to 1 - p
  :param k: The slope of the logistic
  """
  max_n = max - min
  k = - np.log((1 - p )/(p - c)) / max_n
  return c + ((1 - c) / (1 + np.exp(-k * x)))

def brierDecomp(preds, outs):

  brier= 1/len(preds) * sum( (preds-outs)**2 )
  ## bin predictions
  bins = np.linspace(0,1,11)
  binCenters = (bins[:-1] +bins[1:]) /2
  binPredInds = np.digitize(preds,binCenters)
  binnedPreds = bins[binPredInds]

  binTrueFreqs = np.zeros(10)
  binPredFreqs = np.zeros(10)
  binCounts = np.zeros(10)

  for i in range(10):
      idx = (preds >= bins[i]) & (preds < bins[i+1])

      binTrueFreqs[i] = np.sum(outs[idx])/np.sum(idx) if np.sum(idx) > 0 else 0
     # print(np.sum(outs[idx]), np.sum(idx), binTrueFreqs[i])
      binPredFreqs[i] = np.mean(preds[idx]) if np.sum(idx) > 0 else 0
      binCounts[i] = np.sum(idx)

  calibration = np.sum(binCounts * (binTrueFreqs - binPredFreqs) ** 2) / np.sum(binCounts) if np.sum(binCounts) > 0 else 0
  refinement = np.sum(binCounts * (binTrueFreqs *(1 - binTrueFreqs))) / np.sum(binCounts) if np.sum(binCounts) > 0 else 0
  # Compute refinement component
  #refinement = brier - calibration
  return brier, calibration, refinement

def predict(m, trace, relevantData):
  with m:
    predictions = pm.sample_posterior_predictive(trace, var_names=["successP"], return_inferencedata=False, predictions=True, extend_inferencedata=False)
    predictionSuccessChainRuns = predictions["successP"][:,:,0:len(relevantData)]
    predictionsSuccessInstance = np.mean(predictionSuccessChainRuns, (0,1))
    successes = np.array([0 if val < -0.99 else 1 for val in relevantData['finalReward']])

    return predictionsSuccessInstance, successes

In [ ]:
def setupModel(data):
  successes = [0 if val < -0.99 else 1 for val in data['finalReward']]

  abilityMin = {}
  abilityMax = {}

  abilityMin["navAbility"] = 0
  abilityMax["navAbility"] = data['distance'].max()

  abilityMin["visualAcuity"] = 0.0001
  abilityMax["visualAcuity"] = (data['distance']/data['size']).min()

  m = pm.Model()
  with m:
    navAbility = pm.HalfNormal("navAbility", sigma = abilityMax["navAbility"]/2)
    visualAcuity = pm.HalfNormal("visualAcuity", sigma = abilityMax["visualAcuity"]/2)

    goalDist = pm.Data("goalDistance", data["distance"])
    goalSize = pm.Data("goalSize", data["size"])

    navP = pm.Deterministic("navP", logistic_general(navAbility - goalDist, min = abilityMin["navAbility"], max = abilityMax["navAbility"], c = 0, p = 0.99))
    visualP = pm.Deterministic("visualP", logistic_general(np.log(visualAcuity) - np.log(goalDist/goalSize), min = np.log(abilityMin["visualAcuity"]), max = np.log(abilityMax["visualAcuity"]), c = 0, p = 0.99))

    successP = pm.Deterministic("successP", navP * visualP)
    # successP = pm.Deterministic("successP", (navP + visualP)/2) # compensatory relationship

    taskSuccess = pm.Bernoulli("taskSuccess", successP, observed = successes)

  return m, abilityMin, abilityMax, successes

In [ ]:
data = combined_df[(combined_df['pixelInput'] == 4) & (combined_df['navigationNoise'] == 0.0)]
m, abilityMin, abilityMax, successes = setupModel(data=data)
gv = pm.model_graph.model_to_graphviz(m)
gv

In [ ]:
pymc_sample_num = 2000

chains = 4

In [ ]:
results = defaultdict(list)
visual_acuity_list = []
nav_ability_list = []

for p in pixel_sizes:
  for n in noise_levels:
    print(f"Fitting measurement layout for agent with {p}x{p} input and 0.{n} navigation noise.")
    results['pixels'].append(p)
    results['noise'].append(n/10)

    data = combined_df[(combined_df['pixelInput'] == p) & (combined_df['navigationNoise'] == n/10)]
    train_set, test_set = train_test_split(data, test_size = 0.2, random_state = 2024)

    model_train, _, _, _ = setupModel(data=train_set)

    with model_train:
      data_training = pm.sample(pymc_sample_num, target_accept=0.95, chains = chains)

    model_test, _, _, _ = setupModel(data=test_set)

    predictionsSuccessInstance, successes = predict(model_test, data_training, test_set)

    agentBrierScoreSuccess, agentCalibrationSuccess, agentRefinementSuccess = brierDecomp(predictionsSuccessInstance, successes)
    agentAggBrierScoreSuccess, agentAggCalibrationSuccess, agentAggRefinementSuccess = brierDecomp(np.repeat(np.mean(successes), len(successes)), successes)

    results['modelBrier'].append(agentBrierScoreSuccess)
    results['aggBrier'].append(agentAggBrierScoreSuccess)
    results['modelBetter'].append(agentBrierScoreSuccess < agentAggBrierScoreSuccess)
    results['modelCalibration'].append(agentCalibrationSuccess)
    results['aggCalibration'].append(agentAggCalibrationSuccess)
    results['modelRefinement'].append(agentRefinementSuccess)
    results['aggRefinement'].append(agentAggRefinementSuccess)
    results['meanSuccessTest'].append(np.mean(successes))

    model_all, abilityMin, abilityMax, successes = setupModel(data=data)

    with model_train:
      data_all = pm.sample(pymc_sample_num, target_accept=0.95, chains = chains)

    visual_acuity_list.append(data_all['posterior']['visualAcuity'])
    nav_ability_list.append(data_all['posterior']['navAbility'])

    navMean = float(np.mean(data_all['posterior']['navAbility']))
    navStd = float(np.std(data_all['posterior']['navAbility']))
    visualMean = float(np.mean(data_all['posterior']['visualAcuity']))
    visualStd = float(np.std(data_all['posterior']['visualAcuity']))

    results['navigationMean'].append(navMean)
    results['navigationStd'].append(navStd)
    results['visualAcuityMean'].append(visualMean)
    results['visualAcuityStd'].append(visualStd)

    results['meanSuccessAll'].append(np.mean(successes))

results_df = pd.DataFrame(results)

In [ ]:
results_df

In [ ]:
visual_acuity_list_subset = [visual_acuity_list[0], visual_acuity_list[8],visual_acuity_list[50], visual_acuity_list[58], visual_acuity_list[90], visual_acuity_list[98]]

In [ ]:
az.plot_forest(visual_acuity_list_subset,
               model_names=["Pixels: 4, Noise: 0.0", "Pixels: 4, Noise: 0.8", "Pixels: 20, Noise: 0.0", "Pixels: 20, Noise: 0.8", "Pixels: 40, Noise: 0.0", "Pixels: 40, Noise: 0.8"],
               var_names='visualAcuity',
                combined=True,
                hdi_prob=0.95,
                quartiles=False,
                legend=False,
                figsize=(10,10),
                # colors = colours_all,
               )

In [ ]:
nav_ability_list_subset = [nav_ability_list[70], nav_ability_list[73], nav_ability_list[78], nav_ability_list[79], nav_ability_list[90], nav_ability_list[93], nav_ability_list[98], nav_ability_list[99]]

In [ ]:
az.plot_forest(nav_ability_list_subset,
               model_names=["Noise: 0.0, Pixels: 32", "Noise: 0.3, Pixels: 32", "Noise: 0.8, Pixels: 32", "Noise: 0.9, Pixels: 32", "Noise: 0.0, Pixels: 40", "Noise: 0.3, Pixels: 40", "Noise: 0.8, Pixels: 40", "Noise: 0.9, Pixels: 40"],
               var_names='navAbility',
                combined=True,
                hdi_prob=0.95,
                quartiles=False,
                legend=False,
                figsize=(10,10),
                # colors = colours_all,
               )

## Sensitivity Analysis

In [7]:
def setupModelSensitivity(
  data,
  link_func_type="multiplicative", # Options: "multiplicative", "compensatory"
  logistic_p=0.99,                 # The 'p' parameter for logistic_general
  prior="HalfNormal",
  normal_sigma=2,
  generalized_mean_const=1
):
  successes = [0 if val < -0.99 else 1 for val in data['finalReward']]

  abilityMin = {}
  abilityMax = {}

  # 2. Ability/Demand Ranges
  abilityMin["navAbility"] = 0
  abilityMax["navAbility"] = data['distance'].max()
  abilityMin["visualAcuity"] = 0.0001
  abilityMax["visualAcuity"] = (data['distance']/data['size']).min()

  m = pm.Model()
  with m:
    # Priors
    if prior == "HalfNormal":
      navAbility = pm.HalfNormal("navAbility", sigma = abilityMax["navAbility"]/normal_sigma)
      visualAcuity = pm.HalfNormal("visualAcuity", sigma = abilityMax["visualAcuity"]/normal_sigma)
    elif prior == "Uniform":
      assert normal_sigma==2, "Sigma variance has no effect on uniform, so asserting it must be two to avoid duplicate runs."
      navAbility = pm.Uniform("navAbility", lower = abilityMin["navAbility"], upper = (abilityMax["navAbility"] * 1.2)) # add some slack
      visualAcuity = pm.Uniform("visualAcuity", lower = abilityMin["visualAcuity"], upper = (abilityMax["visualAcuity"] * 1.2)) # add some slack
    else:
      raise ValueError(f"Unknown prior: {prior}")

    # 3. Observed Data
    goalDist = pm.Data("goalDistance", data["distance"])
    goalSize = pm.Data("goalSize", data["size"])

    navP = pm.Deterministic("navP", logistic_general(navAbility - goalDist, min = abilityMin["navAbility"], max = abilityMax["navAbility"], c = 0, p = logistic_p))
    visualP = pm.Deterministic("visualP", logistic_general(np.log(visualAcuity) - np.log(goalDist/goalSize), min = np.log(abilityMin["visualAcuity"]), max = np.log(abilityMax["visualAcuity"]), c = 0, p = logistic_p))

    # 4. Linking Function (Sensitivity: link_func_type)
    if link_func_type == "multiplicative":
      assert generalized_mean_const == 1, "Generalized mean parameter must be 1 for multiplicative (non-compensatory) relationship."
      successP = pm.Deterministic("successP", navP * visualP)
    elif link_func_type == "compensatory":
      if generalized_mean_const == 0:
        successP = pm.Deterministic("successP", pm.math.sqrt(navP * visualP))
      else:
        successP = pm.Deterministic("successP", (((navP**generalized_mean_const) + (visualP**generalized_mean_const)) / 2.0)**(1.0/generalized_mean_const))

    # 5. Likelihood
    taskSuccess = pm.Bernoulli("taskSuccess", successP, observed = successes)

  return m, abilityMin, abilityMax, successes

In [ ]:
data = combined_df[(combined_df['pixelInput'] == 20) & (combined_df['navigationNoise'] == 1/10)]
chains = 4
pymc_sample_num = 2000


# Parameters to vary
params = {
    "link_settings": [
        ("compensatory", 0),   # Geometric Mean
        ("compensatory", 1),   # Arithmetic Mean
        ("compensatory", 2),   # Root Mean Square
        ("compensatory", 3),   # Cubic Mean
        ("multiplicative", 1), # Multiplicative baseline (k is ignored/implicit)
    ],
    "logistic_p": [0.95, 0.99, 0.999],
    "priors": ["HalfNormal", "Uniform"],
    "normal_sigmas": [1, 2, 4]
}

results = defaultdict(list)

# Create a list of all permutations
keys, values = zip(*params.items())
permutations = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Filter invalid combinations (e.g., Sigma doesn't apply to Uniform)
valid_scenarios = []
for p in permutations:
    if p['priors'] == 'Uniform' and p['normal_sigmas'] != 2:
        continue # Skip duplicate Uniform runs since sigma doesn't affect them
    valid_scenarios.append(p)

print(f"Total scenarios to run: {len(valid_scenarios)}")

# --- 2. Main Loop ---

for i, scenario in enumerate(valid_scenarios):

    # Unpack scenario settings
    link_type, mean_const = scenario['link_settings']
    p_val = scenario['logistic_p']
    prior_type = scenario['priors']
    sigma_factor = scenario['normal_sigmas']

    print(f"Running Scenario {i+1}: {link_type} (k={mean_const}), p={p_val}, {prior_type}, sigma_div={sigma_factor}")

    train_set, test_set = train_test_split(data, test_size=0.2, random_state=2024)

    model_train, _, _, _ = setupModelSensitivity(
        train_set,
        link_func_type=link_type,
        generalized_mean_const=mean_const,
        logistic_p=p_val,
        prior=prior_type,
        normal_sigma=sigma_factor
    )

    with model_train:
        data_train = pm.sample(pymc_sample_num, target_accept=0.95, chains = chains, random_seed=42)

    model_test, _, _, _ = setupModelSensitivity(
        test_set,
        link_func_type=link_type,
        generalized_mean_const=mean_const,
        logistic_p=p_val,
        prior=prior_type,
        normal_sigma=sigma_factor
    )

    predictionsSuccessInstance, successes = predict(model_test, data_train, test_set)

    agentBrierScoreSuccess, _, _ = brierDecomp(predictionsSuccessInstance, successes)
    agentAggBrierScoreSuccess, _, _ = brierDecomp(np.repeat(np.mean(successes), len(successes)), successes)

    model_all, _, _, _ = setupModelSensitivity(
        data,
        link_func_type=link_type,
        generalized_mean_const=mean_const,
        logistic_p=p_val,
        prior=prior_type,
        normal_sigma=sigma_factor
    )

    with model_all:
      data_all = pm.sample(pymc_sample_num, target_accept=0.95, chains = chains, random_seed=42)

    # Extract Posteriors
    post_nav = data_all.posterior["navAbility"].values.flatten()
    post_vis = data_all.posterior["visualAcuity"].values.flatten()

    # --- 3. Store Results ---
    results['link_type'].append(link_type)
    results['gen_mean_k'].append(mean_const)
    results['logistic_p'].append(p_val)
    results['prior_type'].append(prior_type)
    results['sigma_factor'].append(sigma_factor)

    # Performance
    results['modelBrier'].append(agentBrierScoreSuccess)
    results['aggBrier'].append(agentAggBrierScoreSuccess)
    results['modelBetter'].append(agentBrierScoreSuccess < agentAggBrierScoreSuccess)

    # Parameter Estimates
    results['navMean'].append(np.mean(post_nav))
    results['navStd'].append(np.std(post_nav))
    results['visualMean'].append(np.mean(post_vis))
    results['visualStd'].append(np.std(post_vis))

results_df = pd.DataFrame(results)
print("Sensitivity Analysis Complete.")
# display(results_df)

In [9]:
display(results_df)

,link_type,gen_mean_k,logistic_p,prior_type,sigma_factor,modelBrier,aggBrier,modelBetter,navMean,navStd,visualMean,visualStd
0,compensatory,0,0.950,HalfNormal,1,0.111274,0.144375,True,1.335024,1.264138,0.290527,0.170749
1,compensatory,0,0.950,HalfNormal,2,0.111326,0.144375,True,1.415236,1.294348,0.270207,0.152615
2,compensatory,0,0.950,HalfNormal,4,0.111551,0.144375,True,1.505700,1.315443,0.230598,0.110588
3,compensatory,0,0.950,Uniform,2,0.111225,0.144375,True,1.334879,1.238681,0.296526,0.178505
4,compensatory,0,0.990,HalfNormal,1,0.097202,0.144375,True,4.456546,1.816209,1.875546,0.654955
5,compensatory,0,0.990,HalfNormal,2,0.097875,0.144375,True,5.736436,1.754598,1.202953,0.355829
6,compensatory,0,0.990,HalfNormal,4,0.099212,0.144375,True,7.171383,1.733994,0.744556,0.188477
7,compensatory,0,0.990,Uniform,2,0.097699,0.144375,True,5.667255,1.604200,1.228038,0.212471
8,compensatory,0,0.999,HalfNormal,1,0.082857,0.144375,True,9.996157,1.288723,2.978181,0.662114
9,compensatory,0,0.999,HalfNormal,2,0.084464,0.144375,True,11.288456,1.280076,1.968421,0.355013
